# Этап 2. Набор признаков

Вход: нет

Выход: `data/feature_schema.json`

In [ ]:
%pip install -q "yt-dlp[default]" imageio-ffmpeg rapidfuzz librosa soundfile umap-learn transformers tqdm
%pip install -q deno

In [ ]:
import os, sys, json, time, re, subprocess, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 80)

IN_COLAB = 'google.colab' in sys.modules
USE_DRIVE = True

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/music_dataset')
else:
    BASE_DIR = Path('music_dataset').resolve()

AUDIO_DIR, DATA_DIR, FIG_DIR = BASE_DIR / 'audio', BASE_DIR / 'data', BASE_DIR / 'figures'
for d in (AUDIO_DIR, DATA_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)

GENRE_ORDER = ['rock', 'pop', 'hip_hop', 'electronic', 'jazz', 'classical',
               'metal', 'rnb_soul', 'country', 'reggae', 'ambient', 'latin']
GENRE_RU = {'rock': 'рок', 'pop': 'поп', 'hip_hop': 'хип-хоп', 'electronic': 'электроника',
            'jazz': 'джаз', 'classical': 'классика', 'metal': 'метал', 'rnb_soul': 'R&B / соул',
            'country': 'кантри', 'reggae': 'регги', 'ambient': 'эмбиент', 'latin': 'латино'}

## 2. Какие признаки извлекаю

Признаки я разделила на несколько групп по тому, откуда они берутся.

**1. Аудио-эмбеддинг CLAP (512 чисел).** Плотное описание того, как звучит трек: тембр, аранжировка, продакшн. Его удобно использовать для поиска похожих треков. Главный плюс: эмбеддинг есть у любого трека, даже у нового, по которому ещё нет прослушиваний (проблема холодного старта).

**2. Семантические теги** (получаю zero-shot через CLAP):
- настроение: energetic, calm, happy, sad, aggressive, romantic, dreamy, dark;
- контекст: тренировка, вечеринка, работа/учёба, отдых, сон, в дорогу, романтический вечер;
- атмосфера: эпичная, камерная, грувовая, загадочная, ностальгическая, праздничная;
- инструменты (есть/нет): электрогитара, акустическая гитара, фортепиано, синтезатор, струнные, духовые/саксофон, ударные;
- есть ли вокал.

Эти признаки понятны человеку. По ним можно объяснить рекомендацию («похоже по настроению») и собрать плейлист под ситуацию. Пользователи часто ищут музыку именно так: «что-нибудь бодрое для пробежки», а не по жанру. Метал и электроника могут одинаково подходить для тренировки.

**3. Сигнальные признаки** (librosa): темп, громкость (RMS), динамический диапазон, яркость (спектральный центроид), шумность, плотность атак, выраженность ритма, доля ударных, лад (мажор/минор). Они считаются формулами, без нейросетей, поэтому им можно доверять. Темп и энергия напрямую связаны с контекстами вроде тренировки или сна. Ещё по ним можно проверить, насколько адекватны теги CLAP.

**4. Метаданные:** жанр, год, десятилетие, эпоха, длительность, популярность (логарифм просмотров). Эпоха нужна для тематических подборок вроде «хиты 80-х», популярность — чтобы балансировать выдачу между хитами и менее известными треками. Жанр я задала сама при отборе, поэтому использую его как эталон для проверки, а не как признак.

**5. Похожие артисты.** Считаю по ближайшим трекам в пространстве эмбеддингов (этап 5). Можно дополнительно спросить LLM, но это опционально.

CLAP тоже умеет предсказывать жанр. Я его считаю только чтобы проверить, насколько модель вообще справляется, и в признаки для похожести не включаю: иначе оценка по жанру была бы нечестной.

Для zero-shot у каждого класса по 2–3 текстовых шаблона, их эмбеддинги усредняются. Так результат меньше зависит от конкретной формулировки.

In [ ]:
FEATURE_SCHEMA = {
    'genre_clap': {
        'type': 'multiclass', 'role': 'validation',
        'templates': ['{} music', 'this is a {} song', 'a {} track'],
        'labels': {'rock': 'rock', 'pop': 'pop', 'hip_hop': 'hip hop rap', 'electronic': 'electronic dance',
                   'jazz': 'jazz', 'classical': 'classical orchestral', 'metal': 'heavy metal',
                   'rnb_soul': 'r&b soul', 'country': 'country', 'reggae': 'reggae',
                   'ambient': 'ambient', 'latin': 'latin'},
    },
    'mood': {
        'type': 'multiclass', 'role': 'feature',
        'templates': ['{} music', 'a {} song', 'this song sounds {}'],
        'labels': {'energetic': 'energetic and powerful', 'calm': 'calm and peaceful',
                   'happy': 'happy and cheerful', 'sad': 'sad and melancholic',
                   'aggressive': 'aggressive and angry', 'romantic': 'romantic and tender',
                   'dreamy': 'dreamy and ethereal', 'dark': 'dark and ominous'},
    },
    'context': {
        'type': 'multiclass', 'role': 'feature',
        'templates': ['music for {}', 'a song perfect for {}', 'a playlist for {}'],
        'labels': {'workout': 'a workout at the gym', 'party': 'a dance party',
                   'focus_work': 'focusing on work or studying', 'relaxation': 'relaxing after a long day',
                   'sleep': 'falling asleep', 'road_trip': 'a road trip in the car',
                   'romantic_evening': 'a romantic dinner'},
    },
    'atmosphere': {
        'type': 'multiclass', 'role': 'feature',
        'templates': ['music with an {} atmosphere', 'a song that feels {}'],
        'labels': {'epic': 'epic and cinematic', 'intimate': 'intimate and personal',
                   'groovy': 'groovy and funky', 'mysterious': 'mysterious and hypnotic',
                   'nostalgic': 'nostalgic and retro', 'festive': 'festive and celebratory'},
    },
    'instrument': {
        'type': 'binary', 'role': 'feature',
        'labels': {
            'electric_guitar': ['music with loud electric guitar', 'music without electric guitar'],
            'acoustic_guitar': ['music with acoustic guitar', 'music without acoustic guitar'],
            'piano': ['music with piano', 'music without piano'],
            'synthesizer': ['music with synthesizers', 'music without synthesizers'],
            'strings': ['music with orchestral strings, violins and cellos', 'music without string instruments'],
            'brass_sax': ['music with brass instruments or saxophone', 'music without brass or saxophone'],
            'drums': ['music with a prominent drum beat', 'music without drums'],
        },
    },
    'vocals': {
        'type': 'binary', 'role': 'feature',
        'labels': {'vocals': ['a song with a singer singing vocals', 'instrumental music without vocals']},
    },
}

SIGNAL_FEATURES = {
    'tempo_bpm': 'темп, BPM',
    'rms_mean': 'средняя громкость',
    'rms_dynamic_range_db': 'динамический диапазон, дБ',
    'spectral_centroid_hz': 'яркость (спектральный центроид), Гц',
    'spectral_flatness': 'шумность спектра',
    'onset_rate': 'атак в секунду',
    'pulse_clarity': 'выраженность ритма (0..1)',
    'percussive_ratio': 'доля энергии ударных',
    'zcr_mean': 'zero crossing rate',
    'mode_major': 'мажор (1) / минор (0)',
    'key_strength': 'уверенность в тональности',
}

META_FEATURES = {
    'genre': 'жанр (эталон)', 'year': 'год', 'decade': 'десятилетие', 'era': 'эпоха',
    'duration_sec': 'длительность', 'log_views': 'log10 просмотров', 'is_topic_channel': 'официальный канал лейбла',
}

schema = {'semantic': FEATURE_SCHEMA, 'signal': SIGNAL_FEATURES, 'meta': META_FEATURES,
          'embedding': {'clap_audio': 'laion/larger_clap_music, 512'}}
(DATA_DIR / 'feature_schema.json').write_text(json.dumps(schema, ensure_ascii=False, indent=2), encoding='utf-8')

rows = []
for cat, spec in FEATURE_SCHEMA.items():
    for lab, txt in spec['labels'].items():
        rows.append({'группа': 'семантика (CLAP)', 'признак': cat, 'класс': lab, 'роль': spec['role'],
                     'описание': txt if isinstance(txt, str) else ' / '.join(txt)})
rows += [{'группа': 'сигнал (librosa)', 'признак': k, 'класс': '', 'роль': 'feature', 'описание': v} for k, v in SIGNAL_FEATURES.items()]
rows += [{'группа': 'метаданные', 'признак': k, 'класс': '', 'роль': 'target' if k == 'genre' else 'feature', 'описание': v} for k, v in META_FEATURES.items()]
feature_table = pd.DataFrame(rows)
print('Признаков:', len(feature_table), '+ эмбеддинг')
feature_table